# Embedding extraction!
Uses V-JEPA 2

## Imports

In [ ]:
import torch
import numpy as np
import cv2
import pathlib
import transformers
from pathlib import Path
from tqdm import tqdm
from transformers import AutoVideoProcessor, AutoModel, WhisperModel, AutoFeatureExtractor
from torch.utils.data import Dataset, DataLoader
from accelerate import Accelerator
from accelerate.utils import gather_object
import json
import zipfile
import os
import librosa
import torchaudio.transforms as T

## Unzipping

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

#Configuration
BATCH_SIZE = 4
HF_REPO = "facebook/vjepa2-vitl-fpc64-256"
NUM_WORKERS = 4 #For DataLoader

#Initialize Accelerator:

accelerator = Accelerator()
device = accelerator.device

#we only want to hear from main process
def log(msg):
  if accelerator.is_main_process:
    print(msg)

# ================================= PARAMETER - ADJUST THIS FILEPATH ==============================
zip_file_path = '/content/drive/MyDrive/V-JEPA Twitch Project/Data/dataset_1594760722_clips.zip'
output_dir = Path('./')

if Path(zip_file_path).exists():
    log(f"Unzipping {zip_file_path} to {output_dir}")
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(output_dir)
    log("Unzipping complete.")
else:
    log(f"Zip file not found at {zip_file_path}.")

log(f"Running on {accelerator.num_processes} GPU(s)")
log(f"Device: {device}")

Zip file not found at /content/drive/MyDrive/V-JEPA Twitch Project/Data/dataset_1594760722_clips.zip.
Running on 1 GPU(s)
Device: cuda


# V-JEPA 2 Process

## Class definition

In [ ]:
class VideoClipDataset(Dataset):
  #class to turn video clips into numpy arrays
  def __init__(self, video_dir, embed_dir):
    self.video_dir = Path(video_dir)
    self.embed_dir = Path(embed_dir)
    self.embed_dir.mkdir(parents = True, exist_ok = True)

    all_videos = sorted(self.video_dir.glob("*.mp4"))
    self.video_files = [
        v for v in all_videos
        if not (self.embed_dir / f"{v.stem}.npy").exists()
    ]

  def __len__(self):
    return len(self.video_files)

  def __getitem__(self, idx):
    video_path = self.video_files[idx]
    clip_id = video_path.stem # eg. clip_26xxxxx_00001

    #Load Video Frames
    cap = cv2.VideoCapture(str(video_path))
    frames = []

    while True:
      ret, frame = cap.read()
      if not ret:
        break
      frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
      frames.append(frame_rgb)
    cap.release()

    if len(frames) == 0:
      frames = [np.zeros((256, 256, 3), dtype=np.uint8)] * 64

    video = np.stack(frames)
    return {"video" : video, "clip_id" : clip_id}


def collate_fn(batch, processor):
  videos = [item["video"] for item in batch]
  clip_ids = [item["clip_id"] for item in batch]

  inputs = processor(videos, return_tensors="pt")

  return {
      "pixel_values" : inputs["pixel_values_videos"],
      "clip_ids" : clip_ids
  }


def extract_embeddings_distributed(dataset_folder):
  dataset_folder = Path(dataset_folder)
  video_dir = dataset_folder / "video"

  # Extract ID from dataset_folder name, e.g., 'dataset_2657956036' -> '2657956036'
  dataset_id = dataset_folder.name.split('_')[-1]
  embed_dir = Path(f"./jepa-embeddings_{dataset_id}")

  log(f"Loading V-Jepa 2")

  model = AutoModel.from_pretrained(HF_REPO)
  processor = AutoVideoProcessor.from_pretrained(HF_REPO)
  model.eval()

  dataset = VideoClipDataset(video_dir, embed_dir)
  log(f"Found {len(dataset)} unprocessed clips")

  if len(dataset) == 0:
    log("All clips already processed!")
    return

  dataloader = DataLoader(
      dataset,
      batch_size = BATCH_SIZE,
      shuffle = False,
      num_workers = NUM_WORKERS,
      collate_fn = lambda b: collate_fn(b, processor),
      pin_memory = True
  )

  model, dataloader = accelerator.prepare(model, dataloader)

  processed_count = 0

  progress_bar = tqdm(
    dataloader,
    desc = f"GPU {accelerator.process_index}",
    disable = not accelerator.is_local_main_process
  )

  for batch in progress_bar:
    pixel_values = batch["pixel_values"]
    clip_ids = batch["clip_ids"]

    with torch.no_grad():
      outputs = model(pixel_values_videos=pixel_values)
      embeddings = outputs.last_hidden_state.mean(dim=1)

    embeddings_np = embeddings.cpu().numpy()
    for clip_id, emb in zip(clip_ids, embeddings_np):
      embed_path = embed_dir / f"{clip_id}.npy"
      np.save(embed_path, emb)
      processed_count += 1

  accelerator.wait_for_everyone()
  total_processed = torch.tensor([processed_count], device=device)
  total_processed = accelerator.gather(total_processed).sum().item()

  log(f"\n Processed {int(total_processed)} clips")

## Execution

In [ ]:
dataset_folders = sorted(Path(".").glob("dataset_*"))
log(f"Found {len(dataset_folders)} dataset folders")

for folder in dataset_folders:
  extract_embeddings_distributed(folder)

log("All embeddings extracted!")

Found 0 dataset folders
All embeddings extracted!


# Whisper embeddings


## Class definition

In [ ]:
# Configuration
BATCH_SIZE = 32
HF_REPO_AUDIO = "openai/whisper-small"
NUM_WORKERS = 0  # Set to 0 to avoid multiprocessing crashes in Colab
TARGET_SR = 16000
WHISPER_SAMPLE_LENGTH = 480000  # 30 seconds * 16000 Hz
CLIP_LENGTH = 240000            # 15 seconds * 16000 Hz

accelerator = Accelerator()
device = accelerator.device

class AudioClipDataset(Dataset):
    def __init__(self, audio_dir, embed_dir):
        self.audio_dir = Path(audio_dir)
        self.embed_dir = Path(embed_dir)
        self.embed_dir.mkdir(parents=True, exist_ok=True)

        all_audio = sorted(self.audio_dir.glob("*.mp3"))

        self.audio_files = [
            f for f in all_audio
            if not (self.embed_dir / f"{f.stem}.npy").exists()
        ]

    def __len__(self):
        return len(self.audio_files)

    def __getitem__(self, idx):
        audio_path = self.audio_files[idx]
        clip_id = audio_path.stem

        try:
            audio_array, sr = librosa.load(str(audio_path), sr=None, mono=False)
        except Exception as e:
            print(f"Error loading {audio_path}: {e}")
            return {"audio": np.zeros(WHISPER_SAMPLE_LENGTH), "clip_id": clip_id}

        waveform = torch.from_numpy(audio_array)

        if waveform.ndim == 1:
            waveform = waveform.unsqueeze(0)

        if sr != TARGET_SR:
            resampler = T.Resample(sr, TARGET_SR)
            waveform = resampler(waveform)

        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        audio_array = waveform.squeeze().numpy()

        return {"audio": audio_array, "clip_id": clip_id}

def collate_fn_audio(batch, processor):
    raw_audio = [item["audio"] for item in batch]
    clip_ids = [item["clip_id"] for item in batch]

    padded_audio = []

    for audio in raw_audio:
        curr_len = len(audio)
        if curr_len < WHISPER_SAMPLE_LENGTH:
            pad_len = WHISPER_SAMPLE_LENGTH - curr_len
            audio = np.pad(audio, (0, pad_len), 'constant')
        else:
            audio = audio[:WHISPER_SAMPLE_LENGTH]
        padded_audio.append(audio)

    inputs = processor(
        padded_audio,
        sampling_rate=TARGET_SR,
        return_tensors="pt"
    )

    batch_size = len(clip_ids)
    attention_mask = torch.zeros(batch_size, 1500)
    attention_mask[:, :750] = 1.0

    return {
        "input_features": inputs.input_features,
        "attention_mask": attention_mask,
        "clip_ids": clip_ids
    }

def extract_audio_embeddings_distributed(dataset_folder):
    dataset_folder = Path(dataset_folder)
    audio_dir = dataset_folder / "audio"

    # Standardized root-level directory
    dataset_id = dataset_folder.name.split('_')[-1]
    embed_dir = Path(f"./whisper-embeddings_{dataset_id}")

    log(f"Loading Whisper Model: {HF_REPO_AUDIO}")

    model = WhisperModel.from_pretrained(HF_REPO_AUDIO)
    del model.decoder
    processor = AutoFeatureExtractor.from_pretrained(HF_REPO_AUDIO)

    model.eval()

    dataset = AudioClipDataset(audio_dir, embed_dir)
    log(f"Found {len(dataset)} unprocessed audio clips")

    if len(dataset) == 0:
        return

    dataloader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        collate_fn=lambda b: collate_fn_audio(b, processor),
        pin_memory=True
    )

    model, dataloader = accelerator.prepare(model, dataloader)

    processed_count = 0
    progress_bar = tqdm(
        dataloader,
        desc=f"GPU {accelerator.process_index}",
        disable=not accelerator.is_local_main_process
    )

    for batch in progress_bar:
        input_features = batch["input_features"]
        attention_mask = batch["attention_mask"].to(device)
        clip_ids = batch["clip_ids"]

        with torch.no_grad():
            encoder_outputs = model.encoder(input_features)
            last_hidden_state = encoder_outputs.last_hidden_state

            mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size())
            sum_embeddings = torch.sum(last_hidden_state * mask_expanded, dim=1)
            sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
            embeddings = sum_embeddings / sum_mask

        embeddings_np = embeddings.cpu().numpy()

        for clip_id, emb in zip(clip_ids, embeddings_np):
            embed_path = embed_dir / f"{clip_id}.npy"
            np.save(embed_path, emb)
            processed_count += 1

    accelerator.wait_for_everyone()
    log(f"Finished folder: {dataset_folder.name}")

## Execution

In [ ]:
dataset_folders = sorted(Path(".").glob("dataset_*"))
log(f"Found {len(dataset_folders)} dataset folders")

for folder in dataset_folders:
    extract_audio_embeddings_distributed(folder)

log("All audio embeddings extracted!")

Found 0 dataset folders
All audio embeddings extracted!


# Concatenate embeddings


In [ ]:
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

dataset_folders = sorted(Path(".").glob("dataset_*"))

def concatenate_embeddings(dataset_folder):
    dataset_folder = Path(dataset_folder)
    dataset_id = dataset_folder.name.split('_')[-1]

    # Standardized directories in root
    audio_embed_dir = Path(f"./whisper-embeddings_{dataset_id}")
    video_embed_dir = Path(f"./jepa-embeddings_{dataset_id}")
    combined_embed_dir = Path(f"./embeddings_{dataset_id}")

    combined_embed_dir.mkdir(parents=True, exist_ok=True)

    if not audio_embed_dir.exists():
        print(f"Audio embeddings directory not found: {audio_embed_dir}")
        return

    if not video_embed_dir.exists():
        print(f"Video embeddings directory not found: {video_embed_dir}")
        return

    audio_files = sorted(audio_embed_dir.glob("*.npy"))

    if len(audio_files) == 0:
        print(f"No audio embeddings found in {audio_embed_dir}")
        return

    successful = 0
    missing_video = 0

    print(f"Processing {len(audio_files)} clips from {dataset_folder.name}")

    for audio_file in tqdm(audio_files, desc=f"Concatenating {dataset_folder.name}"):
        clip_id = audio_file.stem
        video_file = video_embed_dir / f"{clip_id}.npy"
        # Update filename to include '_embeddings' suffix
        combined_file = combined_embed_dir / f"{clip_id}_embeddings.npy"

        if combined_file.exists():
            continue

        if not video_file.exists():
            missing_video += 1
            continue

        try:
            audio_emb = np.load(audio_file)
            video_emb = np.load(video_file)

            combined_emb = np.concatenate([audio_emb, video_emb], axis=0)

            np.save(combined_file, combined_emb)
            successful += 1

        except Exception as e:
            print(f"Error processing {clip_id}: {e}")
            continue

    print(f"Finished {dataset_folder.name}:")
    print(f"  - Successfully combined: {successful}")
    print(f"  - Missing video embeddings: {missing_video}")
    print(f"  - Combined embeddings saved to: {combined_embed_dir}")

for folder in dataset_folders:
    concatenate_embeddings(folder)

print("\nAll embeddings concatenated!")


All embeddings concatenated!


# Download embeddings

In [ ]:
import shutil
import os

# Standardized path based on previous cells
dataset_id = dataset_folders[0].name.split('_')[-1]
embed_dir_name = f"embeddings_{dataset_id}"
zip_filename = f"{embed_dir_name}.zip"

if Path(embed_dir_name).exists():
    log(f"Zipping {embed_dir_name} to {zip_filename}")
    # make_archive appends .zip automatically, so we pass the base name
    shutil.make_archive(embed_dir_name, 'zip', embed_dir_name)
    log(f"Successfully created {zip_filename}. You can now download it from the files pane.")
else:
    log(f"Embedding directory {embed_dir_name} not found.")

Zipping embeddings_2657956036 to embeddings_2657956036.zip
Successfully created embeddings_2657956036.zip. You can now download it from the files pane.
